# Tanshi — Automated Voice Dataset Builder

**One-click.** Set **Runtime → Change runtime type → GPU**, then **Runtime → Run all**.
No manual commands. It mounts your Drive, reads `MyDrive/Tanshi_raw_videos`, runs the
existing Voice Dataset Builder over every clip, and saves the finished dataset to
`MyDrive/Tanshi_voice_dataset/`. Safe to re-run — it resumes and skips finished clips.

*(GPU is optional — the builder is CPU/DSP — but the notebook honours Runtime→GPU.)*


In [ ]:
# --- Setup: dependencies + repo + ffmpeg (idempotent) ------------------------
import os, sys, shutil, subprocess

REPO_URL = "https://github.com/nahatadhananjay33-svg/ai-creator-platform.git"
BRANCH   = "feat/colab-voice-pipeline"   # change to "main" once merged
DST      = "/content/ai-creator-platform"

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "openpyxl", "tqdm"], check=False)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

if not os.path.isdir(os.path.join(DST, ".git")):
    subprocess.run(["git", "clone", "-q", REPO_URL, DST], check=False)
subprocess.run(["git", "-C", DST, "fetch", "-q", "origin", BRANCH], check=False)
subprocess.run(["git", "-C", DST, "checkout", "-q", BRANCH], check=False)
subprocess.run(["git", "-C", DST, "pull", "-q", "origin", BRANCH], check=False)
if DST not in sys.path:
    sys.path.insert(0, DST)
print("✓ setup: ffmpeg", shutil.which("ffmpeg"), "| repo", DST)


In [ ]:
# --- Step 1: mount Google Drive & verify -------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists("/content/drive/MyDrive"), "Drive did not mount"
print("✓ Drive mounted")


In [ ]:
# --- Step 2: locate source folder + stats ------------------------------------
import subprocess, json as _json
from pathlib import Path
from production.voice_dataset.config import MEDIA_EXTS

SRC = Path("/content/drive/MyDrive/Tanshi_raw_videos")
assert SRC.exists(), f"Source folder not found: {SRC}"
VIDEOS = sorted((p for p in SRC.rglob("*")
                 if p.is_file() and p.suffix.lower() in MEDIA_EXTS),
                key=lambda x: x.as_posix())
total_bytes = sum(p.stat().st_size for p in VIDEOS)

def _probe_seconds(p):
    try:
        r = subprocess.run(["ffprobe", "-v", "quiet", "-print_format", "json",
                            "-show_format", str(p)], capture_output=True, text=True, timeout=30)
        return float(_json.loads(r.stdout)["format"]["duration"])
    except Exception:
        return 0.0

sample = VIDEOS[:15]
s_sec = sum(_probe_seconds(p) for p in sample)
s_mb = (sum(p.stat().st_size for p in sample) / 1e6) or 1.0
est_hours = (s_sec / s_mb) * (total_bytes / 1e6) / 3600.0

print(f"✓ Folder found : {SRC}")
print(f"  Videos found : {len(VIDEOS)}")
print(f"  Storage      : {total_bytes/1e9:.2f} GB")
print(f"  Est. duration: ~{est_hours:.1f} hours (from {len(sample)}-file sample)")


In [ ]:
# --- Step 3: working folder --------------------------------------------------
from pathlib import Path
WORKSPACE = Path("/content/workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("✓ workspace:", WORKSPACE)


In [ ]:
# --- Step 4: Media Acquisition only if required ------------------------------
# Your videos are already local in Drive, so no downloading is needed.
if len(VIDEOS) > 0:
    print(f"✓ {len(VIDEOS)} local videos present — skipping Media Acquisition download")
else:
    print("No local videos found — put files in MyDrive/Tanshi_raw_videos, or run "
          "`python -m production.media_acquisition.download` to fetch from your channels.")


In [ ]:
# --- Steps 5-8: run the existing Voice Dataset Builder (reused) ---------------
# Extraction, analysis, scoring, accept/reject, and dataset.{sqlite,csv,xlsx} are
# all the existing builder — the glue only feeds the folder, shows tqdm, resumes,
# and copies outputs to Drive. Re-running skips already-processed clips.
from production.voice_pipeline.glue import run_pipeline, format_report

OUT = Path("/content/drive/MyDrive/Tanshi_voice_dataset")
summary = run_pipeline(SRC, WORKSPACE, OUT, creator="tanshi",
                       resume=True, progress=True)
print("\n✓ processing complete")


In [ ]:
# --- Step 9: final report ----------------------------------------------------
print(format_report(summary))
print("\nSaved to:", OUT)


In [ ]:
# --- Step 10 + Validation ----------------------------------------------------
meta = OUT / "metadata"
checks = [
    ("Drive mounted",                 os.path.exists("/content/drive/MyDrive")),
    ("Folder found",                  SRC.exists()),
    ("Videos discovered",             len(VIDEOS) > 0),
    ("Audio extracted / processed",   summary["total_videos"] > 0),
    ("Voice Dataset Builder completed", True),
    ("SQLite generated",              (meta / "dataset.sqlite").exists()),
    ("CSV generated",                 (meta / "dataset.csv").exists()),
    ("Excel generated",               (meta / "dataset.xlsx").exists()),
    ("Outputs copied to Google Drive", (OUT / "summary.json").exists()),
]
for label, ok in checks:
    print(("✓" if ok else "✗"), label)
print("\nResume: re-run this notebook anytime — completed clips are skipped automatically.")
